<a href="https://colab.research.google.com/github/kenleefk-edu/C3669C-2026-05/blob/main/Copy_of_Llama_3_2_FineTuning_MetaMathQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Llama 3.2 1B on MetaMathQA with Unsloth
This notebook demonstrates the process of fine-tuning the Llama 3.2 1B model using the Unsloth library.  
Unsloth is optimized for faster training and lower memory usage.

`C3669C_LEEFOOKKIN_4582883W`

### Assignment Sections:
1. Environment Setup
2. Data Preparation
3. Fine-tuning Implementation
4. Evaluation and Analysis
5. Documentation and Report

## 1. Environment Setup
In this section, we set up the development environment with GPU support and install necessary dependencies.

In [ ]:
## a. Set up development environment with GPU support
## b. Install required dependencies
## c. Document configuration steps

# Check GPU availability - Google Colab provides T4, V100, or A100 GPUs.
# We use torch to verify that the GPU is accessible.
import torch
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu_stats.name}, Total Memory: {gpu_stats.total_memory / 1024**3:.2f} GB')
else:
    print('No GPU found. Please change runtime type to GPU.')

# Install Unsloth and its dependencies.
# Unsloth is 2x faster and uses 70% less memory than standard Hugging Face fine-tuning.
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install datasets
print('Dependencies installed successfully.')

GPU: Tesla T4, Total Memory: 14.56 GB
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kl99tr_9/unsloth_d1e68682d056420b91e5c8fb6223081f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kl99tr_9/unsloth_d1e68682d056420b91e5c8fb6223081f
  Resolved https://github.com/unslothai/unsloth.git to commit eeb49d54b8d801a1ce922fa15f778e2bad205db0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.5.7-py3-none-any.whl size=34469846 sha256=e06c54a1db83dbe29d7e6663865f355353014043c403f521d070b94477bc06f5
  Stored in directory: /tmp/pip-ephem-wheel-cache-qb73smax/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/

## 2. Data Preparation
We will use the `meta-math/MetaMathQA` dataset, which is a large-scale dataset for mathematical reasoning.

In [ ]:
import torch
from datasets import load_dataset

# Install unsloth_zoo if it's missing
!pip install unsloth_zoo
from unsloth import FastLanguageModel

# --- Configuration ---
max_seq_length = 2048 # Maximum sequence length supported by the model
dtype = None          # Auto-detection (Float16 for T4, Bfloat16 for Ampere+)
load_in_4bit = True   # 4-bit quantization significantly reduces VRAM usage (essential for Colab free tier)

# a. Load and preprocess dataset
# MetaMathQA contains ~395k rows. We load a subset (10,000 rows) for faster training in this demonstration.
dataset = load_dataset('meta-math/MetaMathQA', split='train[:10000]')

# b. Implement data cleaning
# We filter out any rows that might have missing queries or responses.
dataset = dataset.filter(lambda x: x['query'] is not None and x['response'] is not None)

# c. Create training/validation splits
# We split the data: 90% for training and 10% for validation to monitor performance.
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset['train']
val_dataset = dataset['test']

# d. Format data appropriately for the LLM model
# We use the Alpaca-style prompt template to structure the math queries.
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = '<|end_of_text|>' # Standard Llama 3 EOS token to prevent infinite generation

def formatting_prompts_func(examples):
    queries = examples['query']
    responses = examples['response']
    texts = []
    for query, response in zip(queries, responses):
        # Format the query and response into the template and add the EOS token
        text = alpaca_prompt.format(query, response) + EOS_TOKEN
        texts.append(text)
    return { 'text' : texts }

# Apply formatting to both splits
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print(f'Training set size: {len(train_dataset)}')
print(f'Validation set size: {len(val_dataset)}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


README.md: 0.00B [00:00, ?B/s]

MetaMathQA-395K.json:   0%|          | 0.00/396M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/395000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training set size: 9000
Validation set size: 1000


### 2e. Justification for dataset choice
**For completeness and as a self-reminder** :  The `MetaMathQA` dataset was chosen because it provides high-quality mathematical reasoning pairs. Fine-tuning on this dataset helps the model learn step-by-step problem-solving logic, which is a critical capability for small models like Llama 3.2 1B. It covers various math levels and rephrased questions, ensuring robustness.

## 3. Fine-tuning Implementation
We load the model and configure LoRA (Low-Rank Adaptation) for efficient parameter updates.

In [ ]:
# --- Load Model ---
# We use the pre-quantized 4-bit version of Llama 3.2 1B from Unsloth.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-1B-bnb-4bit', # The name or path of the pre-trained model to load. No default, must be specified.
    max_seq_length = max_seq_length,              # Maximum sequence length the model can handle. Default varies by model, often 2048 or 4096.
    dtype = dtype,                                # Data type for model weights (e.g., torch.float16, torch.bfloat16). 'None' auto-detects based on GPU. Default is None.
    load_in_4bit = load_in_4bit,                  # Whether to load the model in 4-bit quantization, saving VRAM. Default is False.
)

# --- Configure LoRA Adapters ---
# LoRA allows us to train only a small fraction (1-10%) of the model parameters.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA attention dimension (rank): Controls the number of trainable parameters in LoRA. Higher values allow more complex updates but use more VRAM. Default is 64.
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], # List of module names to apply LoRA to. Default depends on the model architecture.
    lora_alpha = 16,      # LoRA scaling factor: Multiplies the LoRA weights. Default is 16.
    lora_dropout = 0,     # The dropout probability for LoRA layers. Optimized to 0 for Unsloth's performance. Default is 0.05.
    bias = 'none',        # Whether to train the bias terms in LoRA layers. Optimized to 'none' for Unsloth. Default is 'none'.
    use_gradient_checkpointing = 'unsloth', # Reduces VRAM usage by recomputing activations during backpropagation. 'unsloth' uses an optimized version. Default is True.
    random_state = 3407,  # Random seed for reproducibility of LoRA layer initialization. Default is None.
    use_rslora = False,   # Whether to use Rank Stabilized LoRA. Default is False.
    loftq_config = None,  # Configuration for LoftQ initialization. Default is None.
)

==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.7 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

# a. Configure hyperparameters
training_args = TrainingArguments(
    per_device_train_batch_size = 2,  # Batch size per GPU: This sets the number of training examples processed per device in a single forward/backward pass. Default is 8.
    gradient_accumulation_steps = 4,  # Accumulate gradients: This accumulates gradients over multiple mini-batches to simulate a larger effective batch size (per_device_train_batch_size * gradient_accumulation_steps). Default is 1.
    warmup_steps = 10,                # Warmup phase: The number of steps for the learning rate to linearly increase from 0 to its initial value. Default is 0.
    max_steps = 200,                  # Total training steps: The total number of update steps to perform. This overrides num_train_epochs. Default is -1 (no limit).
    learning_rate = 2e-4,             # Learning rate: The initial learning rate for the optimizer. Default is 5e-5.
    fp16 = not torch.cuda.is_bf16_supported(), # Mixed precision training: Uses 16-bit floating point numbers for training to save memory and speed up computation. Automatically set based on BF16 support. Default is False.
    bf16 = torch.cuda.is_bf16_supported(),     # Bfloat16 training: Uses bfloat16 for training, typically available on Ampere+ GPUs. Automatically set based on BF16 support. Default is False.
    logging_steps = 1,                # Logging frequency: The number of update steps between two loggings. Default is 500.
    optim = 'adamw_8bit',             # Optimizer: Specifies the optimizer to use. 'adamw_8bit' is selected for memory efficiency. Default is 'adamw_torch'.
    weight_decay = 0.01,              # Weight decay: The strength of L2 regularization. Default is 0.
    lr_scheduler_type = 'linear',     # Learning rate scheduler: Defines how the learning rate changes over time. 'linear' decays linearly after warmup. Default is 'linear'.
    seed = 3407,                      # Random seed: Sets the random seed for reproducibility. Default is None.
    output_dir = 'outputs',           # Output directory: The directory where model checkpoints and predictions will be saved. Default is './'.
    eval_strategy = 'steps',          # Evaluation strategy: Defines when evaluation is performed. 'steps' evaluates every eval_steps. Default is 'no'.
    eval_steps = 50,                  # Evaluation frequency: The number of update steps between two evaluations when eval_strategy is 'steps'. Default is 500.
    save_strategy = 'steps',          # Save strategy: Defines when checkpointing is performed. 'steps' saves every save_steps. Default is 'steps'.
    save_steps = 50,                  # Save frequency: The number of update steps between two checkpoint savings when save_strategy is 'steps'. Default is 500.
    load_best_model_at_end = True,    # Load best model: Whether to load the best model found during training at the end of training. Default is False.
    metric_for_best_model = 'loss',   # Best model metric: The metric to use to compare models during evaluation (e.g., 'loss', 'accuracy'). Default is 'eval_loss'.
    report_to = 'none',               # Reporting tools: Disables integration with external reporting tools like Weights & Biases or TensorBoard. Default is 'all'.
)

# b. Set up early stopping and checkpoint
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,
    args = training_args,
    # Early stopping stops training if the validation loss doesn't improve for 3 evaluations
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
)

# c. Execute fine-tuning
trainer.train()

# d. Save fine-tuned model
# This saves the LoRA adapters and the tokenizer.
model.save_pretrained('lora_model_final')
tokenizer.save_pretrained('lora_model_final')
print('Model saved successfully.')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/9000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss,Validation Loss
50,0.647095,0.748830
100,0.695282,0.721637
150,0.737321,0.705717
200,0.706686,0.699621


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Model saved successfully.


## 4. Evaluation and Analysis

In [ ]:
# a. Compare pre and post fine-tuning performance
# b. Analyse the outputs (3 examples)

# Switch model to inference mode
FastLanguageModel.for_inference(model)

test_queries = [
    'If x + 5 = 10, what is x?',
    'What is the derivative of x^2?',
    'A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?'
]

for query in test_queries:
    # Prepare input using the same prompt template used in training
    inputs = tokenizer([alpaca_prompt.format(query, '')], return_tensors='pt').to('cuda')

    # Generate response
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    response = tokenizer.batch_decode(outputs)[0]

    print(f'Query: {query}')
    # We strip the prompt part to show only the generated response
    print(f'Generated Response:\n{response.split("### Response:")[1]}')
    print('-' * 30)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

Query: If x + 5 = 10, what is x?
Generated Response:

We are given that $x + 5 = 10$.
Simplifying, we have $x + 5 = 10$.
Subtracting 5 from both sides, we get $x = 5$.
Therefore, x + 5 = 10, so x = 5.
The answer is: 5<|end_of_text|>
------------------------------


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What is the derivative of x^2?
Generated Response:

We can differentiate x^2 using the product rule.
The derivative of x^2 is 2x.
The derivative of x^2 is 2x.<|end_of_text|>
------------------------------
Query: A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?
Generated Response:

To find the distance traveled by the train in 2.5 hours, we need to calculate the distance traveled in 60 miles divided by 60 miles/hour.
60 miles divided by 60 miles/hour = 1 hour
Therefore, the train travels 60 miles in 1 hour.
To find the distance traveled in 2.5 hours, we can multiply the distance traveled in 1 hour by 2.5 hours.
60 miles * 2.5 hours = 150 miles
Therefore, the train travels 150 miles in 2.5 hours.
#### 150
The answer is: 150
------------------------------


### 4c. Discuss challenges and limitations
1. **Hardware Constraints**: The free tier of Google Colab (T4 GPU) has limited VRAM (16GB), which necessitates the use of 4-bit quantization and small batch sizes.
2. **Dataset Size**: We used a 10,000-sample subset for efficiency. A full fine-tuning on all 395k samples would yield better results but requires significantly more time and compute.
3. **Model Size**: Llama 3.2 1B is a very small model. While surprisingly capable, it may hallucinate on extremely complex mathematical proofs that require deep symbolic reasoning.
4. **Performance**:  
4.a First test run with `test_queries` returned obviously erroneous answers, all 3! Independently checked, twice.  
4.b `max_steps` was set to 60, which is a very short training duration for a task like mathematical reasoning. This likely explains the poor performance.  
4.c `eval_steps` was set to 20, might need more frequent evaluation during a longer training period. The inaccurate answers from `test_queries` points to a longer training period is needed to give the model more opportunities to learn from the dataset.  
4.d modify the `max_steps` to `200` and `eval_steps` to `50` in the `TrainingArguments` to allow for more training and evaluation.  
4.e Then Part 3 and 4 were re-run to see if the inference improves.  
4.f After increasing `max_steps` and `eval_steps` it did give correct answers to questions in the `test_queries`.  


**Before adjustment**:  
```
1. Generated Response:
We can write x + 5 = 10 as x = 5 + 10 - 5 = 10.
Therefore, x = 10.  
The answer is: 10<|end_of_text|>  

2. Generated Response:  
To find the derivative of x^2,  
we need to find the derivative of x.  
We can use the quotient rule to find the derivative of x.  
The quotient rule states that the derivative of x is  
x times the derivative of x minus 1.  
So, the derivative of x^2 is x times the derivative of x minus 1.  
To find the derivative of x, we can use the power rule.  
The power rule states that the derivative of x^k is k times the derivative of x.  
So, the derivative of x^2 is 2x times the derivative of x minus 1. Therefore, the derivative  
------------------------------  
(seems incomplete)

3. Query: A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?  
Generated Response:  
In 1 hour, the train travels 60 miles.  
In 2.5 hours, the train travels 60/2.5 = 24 miles.  
Therefore, the train travels 24 miles in 2.5 hours.  
#### 24  
The answer is: 24<|end_of_text|>  
```

**After adjustment**:  
```
Query: If x + 5 = 10, what is x?  

Generated Response:  
We are given that $x + 5 = 10$.
Simplifying, we have $x + 5 = 10$.
Subtracting 5 from both sides, we get $x = 5$.
Therefore, x + 5 = 10, so x = 5.
The answer is: 5<|end_of_text|>
------------------------------

Query: What is the derivative of x^2?

Generated Response:
We can differentiate x^2 using the product rule.  
The derivative of x^2 is 2x.  
The derivative of x^2 is 2x.<|end_of_text|>  
------------------------------  

Query: A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?  

Generated Response:
To find the distance traveled by the train in 2.5 hours, we need to calculate the distance traveled in 60 miles divided by 60 miles/hour.
60 miles divided by 60 miles/hour = 1 hour
Therefore, the train travels 60 miles in 1 hour.
To find the distance traveled in 2.5 hours, we can multiply the distance traveled in 1 hour by 2.5 hours.
60 miles * 2.5 hours = 150 miles
Therefore, the train travels 150 miles in 2.5 hours.
#### 150
The answer is: 150
------------------------------
```

## 5. Documentation and Report
### Implementation Documentation
The code is documented with inline comments explaining parameters such as `learning_rate`, `batch_size`, and `r` (LoRA rank).  
Default values like `lora_dropout=0` are used as per Unsloth's optimizations.  

Expanded comments for clarity and as a self-reminder.
```
# --- Load Model ---
# We use the pre-quantized 4-bit version of Llama 3.2 1B from Unsloth.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-1B-bnb-4bit', # The name or path of the
                                                  # pre-trained model to load. # No default,
                                                  # must be specified.
    max_seq_length = max_seq_length,              # Maximum sequence length
                                                  # the model can handle.
                                                  # Default varies by model,
                                                  # often 2048 or 4096.
    dtype = dtype,                                # Data type for model
                                                  # weights (e.g., torch.
                                                  # float16, torch.bfloat16). # 'None'
                                                  # auto-detects based on GPU.
                                                  # Default is None.
    load_in_4bit = load_in_4bit,                  # Whether to load
                                                  # the model in 4-bit
                                                  # quantization, saving VRAM.
                                                  # Default is False.
)

# --- Configure LoRA Adapters ---
# LoRA allows us to train only a small fraction (1-10%)
# of the model parameters.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA attention dimension (rank):
                          # Controls the number of trainable parameters
                          # in LoRA.
                          # Higher values allow more complex updates
                          # but use more VRAM.
                          # Default is 64.
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
                          # List of module names to apply LoRA to.
                          # Default depends on the model architecture.
    lora_alpha = 16,      # LoRA scaling factor:
                          # Multiplies the LoRA weights.
                          # Default is 16.
    lora_dropout = 0,     # The dropout probability for LoRA layers.
                          # Optimized to 0 for Unsloth's performance.
                          # Default is 0.05.
    bias = 'none',        # Whether to train the bias terms in LoRA layers.
                          # Optimized to 'none' for Unsloth.
                          # Default is 'none'.
    use_gradient_checkpointing = 'unsloth', # Reduces VRAM usage by
                          # recomputing activations during backpropagation.
                          # 'unsloth' uses an optimized version.
                          # Default is True.
    random_state = 3407,  # Random seed for reproducibility of
                          # LoRA layer initialization.
                          # Default is None.
    use_rslora = False,   # Whether to use Rank Stabilized LoRA.
                          # Default is False.
    loftq_config = None,  # Configuration for LoftQ initialization.
                          # Default is None.
)
```
**Adding more verbose comments for clarification**
```
# a. Configure hyperparameters  
training_args = TrainingArguments(  
    per_device_train_batch_size = 2,  # Batch size per GPU:
                                      # This sets the number of training
                                      # examples processed per device in a
                                      # single forward/backward pass.
                                      # Default is 8.  
    gradient_accumulation_steps = 4,  # Accumulate gradients:
                                      # This accumulates gradients over
                                      # multiple mini-batches to simulate a
                                      # larger effective batch size
                                      # (per_device_train_batch_size *
                                      # gradient_accumulation_steps).
                                      # Default is 1.  
    warmup_steps = 10,                # Warmup phase: The number of steps
                                      # the learning rate to linearly increase # from 0 to its initial value.
                                      # Default is 0.  
    max_steps = 200,                  # Total training steps: The total number
                                      # of update steps to perform.
                                      # This overrides num_train_epochs.
                                      # Default is -1 (no limit).  
    learning_rate = 2e-4,             # Learning rate: The initial learning
                                      # rate for the optimizer.
                                      # Default is 5e-5.  
    fp16 = not torch.cuda.is_bf16_supported(), # Mixed precision training:
                                               # Uses 16-bit floating point
                                               # numbers for training to save
                                               # memory and speed up # computation.
                                               # Automatically set based on # BF16 support.
                                               # Default is False.  
    bf16 = torch.cuda.is_bf16_supported(),     # Bfloat16 training: Uses
                                               # bfloat16 for training,
                                               # typically available on Ampere+ # GPUs. Automatically set based # on BF16 support.
                                               # Default is False.  
    logging_steps = 1,                # Logging frequency: The number of
                                      # update steps between two loggings.
                                      # Default is 500.  
    optim = 'adamw_8bit',             # Optimizer: Specifies the optimizer
                                      # to use.
                                      # 'adamw_8bit' is selected for memory # efficiency.
                                      # Default is 'adamw_torch'.  
    weight_decay = 0.01,              # Weight decay: The strength of
                                      # L2 regularization.
                                      # Default is 0.  
    lr_scheduler_type = 'linear',     # Learning rate scheduler: Defines how
                                      # the learning rate changes over time.
                                      # 'linear' decays linearly after warmup. # Default is 'linear'.  
    seed = 3407,                      # Random seed: Sets the random seed
                                      # for reproducibility.
                                      # Default is None.  
    output_dir = 'outputs',           # Output directory: The directory where
                                      # model checkpoints and predictions
                                      # will be saved.
                                      # Default is './'.  
    eval_strategy = 'steps',          # Evaluation strategy: Defines when
                                      # evaluation is performed.
                                      # 'steps' evaluates every eval_steps.
                                      # Default is 'no'.  
    eval_steps = 50,                  # Evaluation frequency: The number of
                                      # update steps between two evaluations
                                      # when eval_strategy is 'steps'.
                                      # Default is 500.  
    save_strategy = 'steps',          # Save strategy: Defines when
                                      # checkpointing is performed.
                                      # 'steps' saves every save_steps.
                                      # Default is 'steps'.  
    save_steps = 50,                  # Save frequency: The number of update
                                      # steps between two checkpoint savings
                                      # when save_strategy is 'steps'.
                                      # Default is 500.  
    load_best_model_at_end = True,    # Load best model: Whether to load
                                      # the best model found during training
                                      # at the end of training.
                                      # Default is False.  
    metric_for_best_model = 'loss',   # Best model metric: The metric to use
                                      # to compare models during evaluation
                                      # (e.g., 'loss', 'accuracy').
                                      # Default is 'eval_loss'.  
    report_to = 'none',               # Reporting tools: Disables integration
                                      # with external reporting tools like
                                      # Weights & Biases or TensorBoard.
                                      # Default is 'all'.  
```
### Training Logs and Metrics
Training logs (loss, learning rate, and step time) are automatically printed by the `SFTTrainer` during the `trainer.train()` execution. The validation loss is checked every 20 steps to ensure the model is not overfitting.
```
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1  
   \\   /|    Num examples = 9,000 | Num Epochs = 1 | Total steps = 200  
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4  
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8  
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592  

 (0.90% trained)
 [200/200 08:07, Epoch 0/1]

Step  Training  Loss Validation Loss
50	  0.647095	0.748830
100	  0.695282	0.721637
150	  0.737321	0.705717
200	  0.706686	0.699621
```